# Experiments

## Setup: Import Libraries and Scripts

In [1]:
import pandas as pd
from IPython.display import display, HTML
import optimize_prompt as opt  # Full optimization script
import optimize_prompt4 as opt4  # Full optimization script
import zero_shot_baseline as zsb  # Zero-shot baseline script

import pickle
import os
from datetime import datetime

# Style for better table display
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_colwidth', None)  # Show full content in cells

# Path for saving experiment runs
RUNS_PICKLE_PATH = 'experiment_runs.pkl'

## Define Experiments

Add/edit experiments here. Each is a dict with:
- `'name'`: A label for the experiment.
- `'script'`: 'optimize' or 'zero_shot'.
- Other keys: Parameters for main() (e.g., generations, model_name).

In [ ]:
experiments = [
              {
        'name': 'Full Optimization 4',
        'script': 'optimize4',
        'generations': 10,
        'pop_size': 6,
        'train_sample_size': 5,
        'test_sample_size': 50,
        'model_name': 'google/gemini-2.5-flash',
        'use_bandit_instr': True,
        'use_bandit_template': True,
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
  
]

experiments_backlog = [
      {
        'name': 'Full Optimization - w META INSTRUCTION 3 & META TEMPLATE 3 & INSTR_STRATEGIES_ORIGINAL w/o early stopping',
        'script': 'optimize',
        'generations': 40,
        'pop_size': 12,
        'train_sample_size': 20,
        'test_sample_size': 1000,
        'model_name': 'google/gemini-2.5-flash',
        'use_bandit_instr': True,
        'use_bandit_template': True,
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
    {
        'name': 'Full Optimization - w META INSTRUCTION 3 & META TEMPLATE 3 & INSTR_STRATEGIES_ORIGINAL w/o early stopping',
        'script': 'zero_shot',
        'test_sample_size': 1000,
        'model_name': 'google/gemini-2.5-flash',
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
        {
        'name': 'Full Optimization - w META INSTRUCTION 3 & META TEMPLATE 3 & INSTR_STRATEGIES_ORIGINAL w/o early stopping - No Bandit',
        'script': 'optimize',
        'generations': 20,
        'pop_size': 4,
        'train_sample_size': 10,
        'test_sample_size': 300,
        'model_name': 'google/gemini-2.5-flash-lite-preview-06-17',
        'use_bandit_instr': False,
        'use_bandit_template': False,
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
        {
        'name': 'Full Optimization - w META INSTRUCTION 3 & META TEMPLATE 3 - No Bandit',
        'script': 'optimize',
        'generations': 40,
        'pop_size': 4,
        'train_sample_size': 10,
        'test_sample_size': 300,
        'model_name': 'google/gemini-2.5-flash-lite-preview-06-17',
        'use_bandit_instr': False,
        'use_bandit_template': False,
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
]

## Run Experiments

This cell runs each experiment and collects results, saving them to a pickle file with a timecode.

In [3]:
results = []

for exp in experiments:
    print(f"\n=== Running Experiment: {exp['name']} ===")
    script = exp.pop('script')  # Remove script key for passing to main
    name = exp.pop('name')  # Remove name for passing to main
    run_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    try:
        if script == 'optimize':
            result = opt.main(**exp)
        elif script == 'optimize4':
            result = opt4.main(**exp)
        elif script == 'zero_shot':
            result = zsb.main(**exp)
        else:
            raise ValueError(f"Unknown script: {script}")
        
        # Flatten metrics for table
        metrics = result['test_metrics']
        flat_result = {
            'Experiment Name': name,
            'Script': script,
            **exp,  # Add back parameters
            'Best Instruction': result['best_instruction'],
            'Best Template': result['best_template'],
            'Sample Size': metrics['sample_size'],
            'Valid Predictions': metrics['valid_predictions'],
            'Total Predictions': metrics['total_predictions'],
            'Accuracy': metrics['accuracy'],
            'Precision': metrics['precision'],
            'Recall': metrics['recall'],
            'F1 Micro': metrics['f1_micro'],
            'F1 Macro': metrics['f1_macro'],
            'Adjusted F1 Macro': metrics['adjusted_f1_macro'],
            'Support (0/1)': f"{metrics['support'].get('0', 0)} / {metrics['support'].get('1', 0)}",
            'Unique y_true': ', '.join(metrics['unique_y_true']),
            'Unique y_pred': ', '.join(metrics['unique_y_pred']),
            'Detailed Report': metrics['detailed_report_string'],  # Full string for details
            'Full Classification Report (Dict)': metrics['classification_report'],  # Raw dict if needed
            'Run Time': run_time
        }
        results.append(flat_result)
    except Exception as e:
        print(f"Error in experiment '{name}': {e}")
        results.append({'Experiment Name': name, 'Error': str(e), 'Run Time': run_time})

# Load previous runs if exists
if os.path.exists(RUNS_PICKLE_PATH):
    with open(RUNS_PICKLE_PATH, 'rb') as f:
        past_runs = pickle.load(f)
else:
    past_runs = []

# Add new results to past runs and save
all_runs = past_runs + results
with open(RUNS_PICKLE_PATH, 'wb') as f:
    pickle.dump(all_runs, f)


=== Running Experiment: Full Optimization 4 ===
============ Generation 1 ============


Evaluating population:   0%|          | 0/2 [00:00<?, ?it/s]

---- Sent in Batch 1 ----
Instruction: Classify the following clause from a Terms of Service contract as fair (0) or unfair (1) using the statutory context and contract context for better understanding. Respond only with '0' or '1'.
Clause: this section will be given full effect even if any remedy specified in these terms is deemed to have failed of its essential purpose .
Statutory Context: According to art. 3 of the Directive 93/13 on Unfair Terms in Consumer Contracts, a contractual term is unfair if: 1) it has not been individually negotiated; and 2) contrary to the requirement of good faith, it causes a significant imbalance in the parties' rights and obligations, to the detriment of the consumer. This general definition is further specified in the Annex to the Directive, containing an indicative and non-exhaustive list of the terms which may be regarded as unfair, as well as in a few dozen judgments of the Court of Justice of the EU (Micklitz and Reich 2014). Examples of unfair c

Evaluating population:  50%|█████     | 1/2 [00:04<00:04,  4.94s/it]

⭐ Adjusted F1 Macro Score: 0.1667
---- Sent in Batch 1 ----
Instruction: Classify the following clause from a Terms of Service contract as fair (0) or unfair (1) using the statutory context and contract context for better understanding. Respond only with '0' or '1'.
Clause: 8.4.3 if defective digital content , which match has supplied , damages a device or digital content belonging to a member or subscriber , and this is caused by match 's failure to use reasonable care and skill , match will either repair the damage or pay him/her compensation .
Statutory Context: According to art. 3 of the Directive 93/13 on Unfair Terms in Consumer Contracts, a contractual term is unfair if: 1) it has not been individually negotiated; and 2) contrary to the requirement of good faith, it causes a significant imbalance in the parties' rights and obligations, to the detriment of the consumer. This general definition is further specified in the Annex to the Directive, containing an indicative and non-ex

...

Evaluating population:  50%|█████     | 1/2 [00:04<00:04,  4.26s/it]

⭐ Adjusted F1 Macro Score: 0.7619
---- Sent in Batch 1 ----
At least Classify the clause: 0 (fair) or 1 (unfair).

Clause: "if you do n't agree with these terms , you must discontinue using the services ."

[IF a. These Terms of Service (“Terms”), along with Opera’s Privacy Statement, form a legally-binding contract between you and Opera Software AS, a Norwegian company whose principal place of business is Gjerdrumsvei 19, 0484, Oslo, Norway, as well as its affiliates (“Opera” and “we,” “us” and “our”). By using the Services (as defined below), you are agreeing to be legally bound by these Terms. If you don’t agree with these Terms, you must discontinue using the Services. EXISTS]
Context from the contract: a. These Terms of Service (“Terms”), along with Opera’s Privacy Statement, form a legally-binding contract between you and Opera Software AS, a Norwegian company whose principal place of business is Gjerdrumsvei 19, 0484, Oslo, Norway, as well as its affiliates (“Opera” and “we,” “u

Evaluating population: 100%|██████████| 2/2 [00:08<00:00,  4.16s/it]

⭐ Adjusted F1 Macro Score: 0.7619
⭐⭐ Scores: [0.7619047619047619, 0.7619047619047619]
Mutating instruction with strategy: Append a brief phrase evoking positive emotion (e.g., 'Achieve outstanding success!') to motivate the model. Focus on: 1) Targeting encouragement or reassurance; 2) Using supportive words like 'excellent' or 'believe'; 3) Emphasizing with exclamation or capitals; 4) Boosting self-esteem via motivational cues. Keep it concise to avoid lengthening the prompt.
Prompt: 
You are an expert prompt engineer gently applying the following transformation strategy to improve an instruction for a legal classification task (predicting the fairness of an individual clause from a ToS contract). It is important that responses at all times only consist of '0' for fair or '1' for unfair.

STRATEGY: 
Append a brief phrase evoking positive emotion (e.g., 'Achieve outstanding success!') to motivate the model. Focus on: 1) Targeting encouragement or reassurance; 2) Using supportive words 

Mutating template with strategy: Enhance the description of relationships between template elements, for example explaining how the statutory context provides legal foundations, the contract context offers specific background, the instruction guides the process, and the clause is the target for classification.
Prompt: 
You are an expert prompt engineer gently applying the following transformation strategy to improve a prompt template for a legal classification task (predicting the fairness of an individual clause from a ToS contract). Ensure the template includes the placeholders: At least <instruction> for the classification instruction and <clause> for the clause text. <contract_context> and <statutory_context> may or may not be part of the template. All placeholders in brackets automatically get replaced by the actual data. It is important that the template does not interfere with the model responding only with '0' for fair and '1' for unfair for the classification task the template

## Display Results Table

Interactive table with all parameters and metrics. Sorted by most recent run.

In [4]:
# Load all runs from pickle and display sorted by most recent run time
import pickle
import pandas as pd
from IPython.display import display, HTML

RUNS_PICKLE_PATH = 'experiment_runs.pkl'

if os.path.exists(RUNS_PICKLE_PATH):
    with open(RUNS_PICKLE_PATH, 'rb') as f:
        all_runs = pickle.load(f)
    # Sort by 'Run Time' descending
    all_runs_sorted = sorted(all_runs, key=lambda x: x.get('Run Time', ''), reverse=True)
    df_results = pd.DataFrame(all_runs_sorted)
    if not df_results.empty:
        styled_df = df_results.style.set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap'}).set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'left')]}
        ]).background_gradient(cmap='viridis', subset=['Adjusted F1 Macro'])
        display(HTML("<h3>All Experiment Runs (Most Recent First)</h3>"))
        display(styled_df)
    else:
        print("No results to display.")
else:
    print("No experiment runs found.")

,Experiment Name,Script,generations,pop_size,train_sample_size,test_sample_size,model_name,use_bandit_instr,use_bandit_template,statutory_context_enabled,contract_context_enabled,Best Instruction,Best Template,Sample Size,Valid Predictions,Total Predictions,Accuracy,Precision,Recall,F1 Micro,F1 Macro,Adjusted F1 Macro,Support (0/1),Unique y_true,Unique y_pred,Detailed Report,Full Classification Report (Dict),Run Time,Error
0,Full Optimization 4,optimize4,3.000000,2.000000,5.000000,50.000000,google/gemini-2.5-flash,True,True,True,True,Classify the following clause from a Terms of Service contract as fair (0) or unfair (1) using the statutory context and contract context for better understanding. Respond only with '0' or '1'.,Instruction: Clause: Statutory Context: Contract Context:,50.000000,50.000000,50.000000,0.880000,0.333333,1.000000,0.880000,0.715909,0.715909,47.0 / 3.0,"1, 0","1, 0",precision recall f1-score support 0 1.0000 0.8723 0.9318 47 1 0.3333 1.0000 0.5000 3 accuracy 0.8800 50 macro avg 0.6667 0.9362 0.7159 50 weighted avg 0.9600 0.8800 0.9059 50,"{'0': {'precision': 1.0, 'recall': 0.8723404255319149, 'f1-score': 0.9318181818181818, 'support': 47.0}, '1': {'precision': 0.3333333333333333, 'recall': 1.0, 'f1-score': 0.5, 'support': 3.0}, 'accuracy': 0.88, 'macro avg': {'precision': 0.6666666666666666, 'recall': 0.9361702127659575, 'f1-score': 0.7159090909090908, 'support': 50.0}, 'weighted avg': {'precision': 0.96, 'recall': 0.88, 'f1-score': 0.9059090909090908, 'support': 50.0}}",2025-08-04 08:04:39,nan
1,Full Optimization 4,optimize4,3.000000,2.000000,5.000000,50.000000,google/gemini-2.5-flash,True,True,True,True,Classify the clause: 0 (fair) or 1 (unfair).,"``` At least Here is the clause to classify: Your task is to classify this clause as either fair or unfair. Respond with '0' if the clause is fair, and '1' if the clause is unfair. ```",50.000000,50.000000,50.000000,0.780000,0.214286,1.000000,0.780000,0.610206,0.610206,47.0 / 3.0,"0, 1","0, 1",precision recall f1-score support 0 1.0000 0.7660 0.8675 47 1 0.2143 1.0000 0.3529 3 accuracy 0.7800 50 macro avg 0.6071 0.8830 0.6102 50 weighted avg 0.9529 0.7800 0.8366 50,"{'0': {'precision': 1.0, 'recall': 0.7659574468085106, 'f1-score': 0.8674698795180723, 'support': 47.0}, '1': {'precision': 0.21428571428571427, 'recall': 1.0, 'f1-score': 0.35294117647058826, 'support': 3.0}, 'accuracy': 0.78, 'macro avg': {'precision': 0.6071428571428571, 'recall': 0.8829787234042553, 'f1-score': 0.6102055279943303, 'support': 50.0}, 'weighted avg': {'precision': 0.952857142857143, 'recall': 0.78, 'f1-score': 0.8365981573352234, 'support': 50.0}}",2025-08-04 08:02:18,nan
2,Full Optimization 4,optimize4,3.000000,2.000000,5.000000,50.000000,google/gemini-2.5-flash,True,True,True,True,Classify the following clause from a Terms of Service contract as fair (0) or unfair (1) using the statutory context and contract context for better understanding. Respond only with '0' or '1'.,Instruction: Clause: Statutory Context: Contract Context:,50.000000,50.000000,50.000000,0.880000,0.333333,1.000000,0.880000,0.715909,0.715909,47.0 / 3.0,"1, 0","1, 0",precision recall f1-score support 0 1.0000 0.8723 0.9318 47 1 0.3333 1.0000 0.5000 3 accuracy 0.8800 50 macro avg 0.6667 0.9362 0.7159 50 weighted avg 0.9600 0.8800 0.9059 50,"{'0': {'precision': 1.0, 'recall': 0.8723404255319149, 'f1-score': 0.9318181818181818, 'support': 47.0}, '1': {'precision': 0.3333333333333333, 'recall': 1.0, 'f1-score': 0.5, 'support': 3.0}, 'accuracy': 0.88, 'macro avg': {'precision': 0.6666666666666666, 'recall': 0.9361702127659575, 'f1-score': 0.7159090909090908, 'support': 50.0}, 'weighted avg': {'precision': 0.96, 'recall': 0.88, 'f1-score': 0.9059090909090908, 'support': 50.0}}",2025-08-04 07:59:02,nan
3,Full Optimization 4,optimize4,10.000000,2.000000,5.000000,50.000000,google/gemini-2.5-flash,True,True,True,True,"You are a legal AI expert. Classify the fairness of a ToS contract clause. Output '0' for fair, '1' for